In [1]:
# ==============================================================================
# 修復任務 1: 訓練並儲存部署用 MacBERT (best_macbert_model.bin)
# ⚠️ 執行前請確認已安裝 my_utils.py (若無請往前翻找並建立)
# ==============================================================================
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import BertTokenizer
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import multiprocessing
# 匯入自定義模組
from my_utils import TextDataset, BertClassifier

# --- Config ---
MODEL_NAME = "hfl/chinese-macbert-base"
MAX_LEN = 256
BATCH_SIZE = 8
EPOCHS = 2
LR = 2e-5
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

print(f"🚀 [Recovery 1] 訓練部署用 MacBERT (Device: {DEVICE})")

# 1. 讀取資料 (包含情緒特徵與內文的那個版本)
if os.path.exists('News_Data_with_Emotion_Features.csv'):
    df = pd.read_csv('News_Data_with_Emotion_Features.csv')
else:
    print("❌ 找不到 News_Data_with_Emotion_Features.csv，請先確認 Phase 2 是否執行過。")
    raise SystemExit

# 確保 text_input
if 'text_input' not in df.columns:
    df['text_input'] = df['title_clean'].astype(str) + " [SEP] " + df['content'].astype(str).str[:200]

# 2. 80/20 切分 (我們用 80% 的資料來訓練這個最終模型)
X_train, _, y_train, _ = train_test_split(
    df['text_input'].values, df['label'].values, 
    test_size=0.2, random_state=42, stratify=df['label'].values
)

print(f"📦 使用 {len(X_train)} 筆資料進行最終訓練...")

# 3. 訓練模型
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
train_ds = TextDataset(X_train, y_train, tokenizer, MAX_LEN)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)

model = BertClassifier(MODEL_NAME, 2).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LR)
scheduler = get_linear_schedule_with_warmup(optimizer, int(len(train_loader)*0.1), len(train_loader)*EPOCHS)
loss_fn = torch.nn.CrossEntropyLoss().to(DEVICE)

model.train()
for epoch in range(EPOCHS):
    loop = tqdm(train_loader, leave=False, desc=f"Ep {epoch+1}/{EPOCHS}")
    for batch in loop:
        ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        lbls = batch['labels'].to(DEVICE)
        
        optimizer.zero_grad()
        out = model(ids, mask)
        loss = loss_fn(out, lbls)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

# 4. 存檔 (這就是 App 要用的檔案！)
torch.save(model.state_dict(), 'best_macbert_model.bin')
print("✅ [成功] 已儲存 best_macbert_model.bin")

🚀 [Recovery 1] 訓練部署用 MacBERT (Device: mps)
📦 使用 18821 筆資料進行最終訓練...


Ep 1/2:   0%|          | 0/2353 [00:00<?, ?it/s]

Ep 2/2:   0%|          | 0/2353 [00:00<?, ?it/s]

✅ [成功] 已儲存 best_macbert_model.bin


In [1]:
# ==============================================================================
# 修復任務 2: 重建 Stacking 模型並匯出 (meta_model.joblib)
# ==============================================================================
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
import joblib
import os

print(f"🚀 [Recovery 2] 重建 Stacking 模型與特徵列表")

# 1. 檢查 Cache 檔是否存在
required_files = ['cache_L1_train.npy', 'cache_L1_test.npy', 'cache_y_train.npy', 'cache_y_test.npy', 'cache_meta_train.pkl', 'cache_meta_test.pkl']
for f in required_files:
    if not os.path.exists(f):
        print(f"❌ 缺少檔案: {f}")
        print("💡 請重新執行 [Phase 3] 產生這些緩存檔。")
        raise SystemExit

# 2. 讀取緩存
L1_train = np.load('cache_L1_train.npy')
y_train = np.load('cache_y_train.npy')
X_train_meta = pd.read_pickle('cache_meta_train.pkl')

# 3. 組合特徵 (MacBERT 分數 + Meta 特徵)
X_train_meta['macbert_score'] = L1_train

# 確保型態正確
for col in X_train_meta.columns:
    X_train_meta[col] = pd.to_numeric(X_train_meta[col], errors='coerce')

# 4. 訓練 Meta Learner
print("🏋️ 正在訓練 HistGradientBoosting...")
meta_model = HistGradientBoostingClassifier(
    learning_rate=0.03, max_iter=500, max_depth=6,
    l2_regularization=0.1, early_stopping=True,
    class_weight='balanced', random_state=42
)
meta_model.fit(X_train_meta, y_train)

# 5. 匯出所有必要檔案
print("💾 正在匯出整合包...")

# A. 模型本體
joblib.dump(meta_model, 'meta_model.joblib')

# B. 特徵欄位順序 (App 預測時順序必須完全一致)
feature_columns = list(X_train_meta.columns)
joblib.dump(feature_columns, 'feature_columns.joblib')

# C. 檢查 Label Encoder (情緒用)
if os.path.exists('emotion_le.pkl'):
    print("   - emotion_le.pkl ... OK")
else:
    print("   ⚠️ emotion_le.pkl 遺失，請重新執行 Phase 1 的最後一段存檔程式碼。")

print("-" * 30)
print("✅ 全部完成！您現在可以去寫 Streamlit 了。")
print("📂 請確認目錄下有：")
print("   1. best_emotion_model.bin")
print("   2. best_macbert_model.bin (剛剛產生的)")
print("   3. meta_model.joblib (剛剛產生的)")
print("   4. feature_columns.joblib (剛剛產生的)")
print("   5. emotion_le.pkl")

🚀 [Recovery 2] 重建 Stacking 模型與特徵列表
🏋️ 正在訓練 HistGradientBoosting...
💾 正在匯出整合包...
   - emotion_le.pkl ... OK
------------------------------
✅ 全部完成！您現在可以去寫 Streamlit 了。
📂 請確認目錄下有：
   1. best_emotion_model.bin
   2. best_macbert_model.bin (剛剛產生的)
   3. meta_model.joblib (剛剛產生的)
   4. feature_columns.joblib (剛剛產生的)
   5. emotion_le.pkl
